#### Librerias

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib

Mounted at /content/drive


#### Variables

In [2]:
MODELOS_PATH = "/content/drive/MyDrive/Trabajo práctico 3/modelos"

#### Carga de matrices y labels

In [3]:
X_train_bow = sparse.load_npz(f"{MODELOS_PATH}/X_train_bow.npz")
X_val_bow = sparse.load_npz(f"{MODELOS_PATH}/X_val_bow.npz")
X_test_bow = sparse.load_npz(f"{MODELOS_PATH}/X_test_bow.npz")

X_train_tfidf = sparse.load_npz(f"{MODELOS_PATH}/X_train_tfidf.npz")
X_val_tfidf = sparse.load_npz(f"{MODELOS_PATH}/X_val_tfidf.npz")
X_test_tfidf = sparse.load_npz(f"{MODELOS_PATH}/X_test_tfidf.npz")

y_train = np.load(f"{MODELOS_PATH}/y_train.npy")
y_val = np.load(f"{MODELOS_PATH}/y_val.npy")
y_test = np.load(f"{MODELOS_PATH}/y_test.npy")

X_train_bow.shape, X_val_bow.shape, X_test_bow.shape

((1360000, 20000), (240000, 20000), (498, 20000))

#### Test binario para evaluación

El train (y por lo tanto la validación, que sale de partir el train) solo tiene
clases 0 y 4 — nunca tuvo neutrales. El test manual sí tiene 3 clases, así que
ese es el único conjunto que hay que filtrar.

In [4]:
mask_binario = y_test != 2

X_test_bow_bin = X_test_bow[mask_binario]
X_test_tfidf_bin = X_test_tfidf[mask_binario]
y_test_bin = y_test[mask_binario]

X_val_bow.shape[0], y_test_bin.shape[0]

(240000, 359)

La validación (X_val) ya es binaria de por sí (~240 mil tweets) porque viene de
partir el training, que nunca tuvo clase neutral. El test manual, filtrado, queda
en 359 tweets binarios. Se evalúa contra las dos: la validación da un número
estadísticamente sólido (mucho volumen), el test da la comparación contra datos
etiquetados de forma distinta (a mano), que es lo que pide la consigna.

#### Modelo 1 — Naive Bayes con Bag of Words

In [5]:
modelo_nb = MultinomialNB()
modelo_nb.fit(X_train_bow, y_train)

pred_nb_val = modelo_nb.predict(X_val_bow)
print("=== Validación (240k tweets) ===")
print(classification_report(y_val, pred_nb_val, target_names=["Negativo", "Positivo"]))

pred_nb = modelo_nb.predict(X_test_bow_bin)
print("=== Test manual (359 tweets) ===")
print(classification_report(y_test_bin, pred_nb, target_names=["Negativo", "Positivo"]))

=== Validación (240k tweets) ===
              precision    recall  f1-score   support

    Negativo       0.77      0.78      0.77    120000
    Positivo       0.77      0.76      0.77    120000

    accuracy                           0.77    240000
   macro avg       0.77      0.77      0.77    240000
weighted avg       0.77      0.77      0.77    240000

=== Test manual (359 tweets) ===
              precision    recall  f1-score   support

    Negativo       0.80      0.83      0.82       177
    Positivo       0.83      0.80      0.82       182

    accuracy                           0.82       359
   macro avg       0.82      0.82      0.82       359
weighted avg       0.82      0.82      0.82       359



Contra la validación grande, Naive Bayes da un número mucho más confiable que
contra el test manual — con 240 mil tweets, el resultado no depende de qué 359
tweets puntuales tocaron en la muestra. Si los dos números son parecidos, es una
buena señal de que el modelo generaliza bien más allá del archivo de test chico.

#### Modelo 2 — Logistic Regression con TF-IDF

In [6]:
modelo_lr = LogisticRegression(max_iter=1000)
modelo_lr.fit(X_train_tfidf, y_train)

pred_lr_val = modelo_lr.predict(X_val_tfidf)
print("=== Validación (240k tweets) ===")
print(classification_report(y_val, pred_lr_val, target_names=["Negativo", "Positivo"]))

pred_lr = modelo_lr.predict(X_test_tfidf_bin)
print("=== Test manual (359 tweets) ===")
print(classification_report(y_test_bin, pred_lr, target_names=["Negativo", "Positivo"]))

=== Validación (240k tweets) ===
              precision    recall  f1-score   support

    Negativo       0.79      0.76      0.78    120000
    Positivo       0.77      0.80      0.79    120000

    accuracy                           0.78    240000
   macro avg       0.78      0.78      0.78    240000
weighted avg       0.78      0.78      0.78    240000

=== Test manual (359 tweets) ===
              precision    recall  f1-score   support

    Negativo       0.80      0.81      0.80       177
    Positivo       0.81      0.80      0.81       182

    accuracy                           0.81       359
   macro avg       0.80      0.81      0.81       359
weighted avg       0.81      0.81      0.81       359



Mismo patrón que con Naive Bayes: la validación grande da el número de referencia,
el test manual la comparación contra datos etiquetados distinto.

#### Evaluación sobre train y test — chequeo temprano de overfitting

In [7]:
pred_nb_train = modelo_nb.predict(X_train_bow)
pred_lr_train = modelo_lr.predict(X_train_tfidf)

print("=== NAIVE BAYES ===")
print("--- Train ---")
print(classification_report(y_train, pred_nb_train, target_names=["Negativo", "Positivo"]))
print("--- Validación ---")
print(classification_report(y_val, pred_nb_val, target_names=["Negativo", "Positivo"]))
print("--- Test manual ---")
print(classification_report(y_test_bin, pred_nb, target_names=["Negativo", "Positivo"]))

print("\n=== LOGISTIC REGRESSION ===")
print("--- Train ---")
print(classification_report(y_train, pred_lr_train, target_names=["Negativo", "Positivo"]))
print("--- Validación ---")
print(classification_report(y_val, pred_lr_val, target_names=["Negativo", "Positivo"]))
print("--- Test manual ---")
print(classification_report(y_test_bin, pred_lr, target_names=["Negativo", "Positivo"]))

=== NAIVE BAYES ===
--- Train ---
              precision    recall  f1-score   support

    Negativo       0.77      0.78      0.78    680000
    Positivo       0.78      0.77      0.77    680000

    accuracy                           0.77   1360000
   macro avg       0.77      0.77      0.77   1360000
weighted avg       0.77      0.77      0.77   1360000

--- Validación ---
              precision    recall  f1-score   support

    Negativo       0.77      0.78      0.77    120000
    Positivo       0.77      0.76      0.77    120000

    accuracy                           0.77    240000
   macro avg       0.77      0.77      0.77    240000
weighted avg       0.77      0.77      0.77    240000

--- Test manual ---
              precision    recall  f1-score   support

    Negativo       0.80      0.83      0.82       177
    Positivo       0.83      0.80      0.82       182

    accuracy                           0.82       359
   macro avg       0.82      0.82      0.82       359
w

Con validación y test, el patrón es más confiable que comparando solo train vs
test chico. Si accuracy_val queda cerca de accuracy_train (sin caída fuerte), no
hay overfitting — el modelo generaliza bien al mismo tipo de datos que nunca vio
durante el entrenamiento. La comparación contra test (etiquetado a mano, distinto
origen) es la prueba adicional que pide la consigna.

#### Guardado de modelos baseline

In [8]:
joblib.dump(modelo_nb, f"{MODELOS_PATH}/modelo_nb_baseline.pkl")
joblib.dump(modelo_lr, f"{MODELOS_PATH}/modelo_lr_baseline.pkl")

['/content/drive/MyDrive/Trabajo práctico 3/modelos/modelo_lr_baseline.pkl']